# Genetic Algorithm for Feature Selection

In [11]:
#import libraries
import pandas as pd
import numpy as np

from sklearn.metrics import accuracy_score

from matplotlib import pyplot as plt

from sklearn.ensemble import RandomForestClassifier

from random import randint

from sklearn.metrics import balanced_accuracy_score,f1_score,precision_score, recall_score
from sklearn.metrics import cohen_kappa_score,matthews_corrcoef

In [2]:
#read input files
def readfiles(trainfile,testfile,valfile):
    print('reading files')
    traindf=pd.read_csv(trainfile)
    Xtrain=traindf.iloc[:,0:-1].values
    ytrain=traindf.iloc[:,-1].values

    testdf=pd.read_csv(testfile)
    Xtest=testdf.iloc[:,0:-1].values
    ytest=testdf.iloc[:,-1].values
    
    valdf=pd.read_csv(valfile)
    Xval=valdf.iloc[:,0:-1].values
    yval=valdf.iloc[:,-1].values
    
    featurelist=traindf.columns[0:-1]

    dimensions=Xtrain.shape[1]
    return Xtrain,Xtest,Xval,ytrain,ytest,yval,dimensions,featurelist

In [3]:
#to generate initial population
def generate_population(pop_size,dimensions):
  
    """
    n_particles,dimensions,options
        
    """
    #create the particles. since this is a feature selection problem,
    #we will use a binary encoding.
    #a group of selected features is called a chromosome
    #each chromosome will have values made up of 0s and 1s
    #1 means the feature has been selected , 0 means feature is not selected
    population=np.random.randint(2,size=(pop_size,dimensions))  #generators.py file
    
    return population


In [4]:
#Select individuals for reproduction. 
def selection(population,scores,k=3):
    #using tournament selection
    pop_size=population.shape[0]
    selection_ix=np.random.randint(pop_size)

    for ix in np.random.randint(0,pop_size,k-1):
        if scores[ix] > scores[selection_ix]:
            selection_ix=ix
        elif scores[ix]== scores[selection_ix]:
            if np.count_nonzero( population[ix]) < np.count_nonzero( population[selection_ix]):
                selection_ix=ix
    return selection_ix

    return selected_population,selected_scores

In [5]:
#crossover
def crossover(p1,p2,r_cross):

    pt=int(r_cross * len(p1))

    c1=np.concatenate((p1[:pt],p2[pt:]))
    c2=np.concatenate((p2[:pt],p1[pt:]))

    return c1,c2

In [6]:
#mutation
def mutation(chromosome,r_mut):
    mutated=chromosome.copy()
    
    for i in range(chromosome.shape[0]):
        if np.random.rand() < r_mut:
            mutated[i]=int(not chromosome[i])
    if np.count_nonzero(mutated)>0:
        return mutated
    else:
        return chromosome


In [7]:
#evaluate the chromosome/feature subset
def evaluate_chromosome(chromosome,classifier,Xtrain,Xtest,ytrain,ytest):
    subset_train=Xtrain[:,chromosome]
    subset_test=Xtest[:,chromosome]
        
    classifier.fit(subset_train,ytrain)
    ypred=classifier.predict(subset_test)

    score=f1_score(ytest,ypred,average='weighted')
        
    return score

In [9]:

#Main Algol to start and run the 

def GA(trainfile,testfile,valfile,n_iters=10,pop_size=20,r_cross,r_mut):
    Xtrain,Xtest,Xval,ytrain,ytest,yval,dimensions,featurelist=readfiles(trainfile,testfile,valfile)
    
    population=generate_population(pop_size,dimensions)
   
    classifier=RandomForestClassifier(random_state=20)

    scores=[ evaluate_chromosome(population[i],classifier,Xtrain,Xval,ytrain,yval) for i in range(pop_size) ]

    scores=np.array(scores)
       
    for iter_ in range(n_iters):

        #selection
        selected=[selection(population,scores) for _ in range(pop_size)]
        selected_chroms=population[selected,:]
        scores=scores[selected]

        #crossover
        crossover_res=[crossover(selected_chroms[i],selected_chroms[i+1],r_cross) for i in range(0,selected_chroms.shape[0],2)]
        
        crossover_result=[]
        for chroms in crossover_res:
            crossover_result.extend(chroms)

        #mutation
        mutation_result=[mutation(chromosome,r_mut) for chromosome in crossover_result]
        population=np.array(mutation_result)

        scores==[evaluate_chromosome(population[i],classifier,Xtrain,Xval,ytrain,yval) for i in range(pop_size)]
        scores=np.array(scores)

        #get best chromosome/subset in current population
        best=np.argmax(scores)
        best_chromosome=population[best,:]
        best_score=scores[best]

        print('generation {} of {} : {}, {}'.format(iter_, n_iters,best_score,np.count_nonzero(best_chromosome)))


    #evaluate overall best chromosome/subset with the test subset
    test_score=evaluate_chromosome(best_chromosome,classifier,Xtrain,Xtest,ytrain,ytest)

    print('score of best subset on the test dataset:',test_score)

In [ ]:

#input files
data_path="https://raw.githubusercontent.com/vappiah/Machine-Learning-Tutorials/refs/heads/main/datasets/Wine"
trainfile='%s/train.csv'%data_path
testfile='%s/test.csv'%data_path
valfile='%s/val.csv'%data_path

#parameters
r_cross=0.7
r_mut=0.2
pop_size=20
n_iters=10


#run algorithm
GA(trainfile,testfile,valfile,n_iters,pop_size,r_cross,r_mut)